# Supply Chain Disruptor — baseline pipeline

Generated from the project modules by `build_notebook.py`. The cells below
write each module to `/kaggle/working`, then run the pipeline end to end and
leave `submission.csv` where Kaggle scores it.

**Where the points are.** Weighted through to final placement, the simulator
playing your orders is 56%, the forecasts are 24%, the video is 20%.

The submission is built by `strategy.py` from estimates `signals.py` fits on
the released weeks. Backtested over six origins against the real supply
history it beats the provided baseline on every component: total cost -15.9%,
stockout cost -61%, service 0.725 -> 0.907, mean HHI 1.000 -> 0.406, Brier
0.1809 -> 0.1770.

**Rerun after Saturday noon** by running all cells. Nothing is hardcoded to
week 208: the submission shape is read off `sample_submission.csv` and the
data directory is discovered under `/kaggle/input/`.


In [ ]:
!nvidia-smi || echo "no GPU attached - correct for this workload"

## Modules

Written verbatim from the project sources.


In [ ]:
%%writefile config.py
"""Tunable constants and paths.

Everything the Saturday-noon rerun touches lives here, so a rerun is a path
change and nothing else. Constants marked CONFIRM are placeholders until you
check them against the competition's own description -- `audit()` lists them.
"""

from dataclasses import dataclass, field
from pathlib import Path
import os

PROJECT_DIR = Path(__file__).resolve().parent

# Kaggle mounts competition data under /kaggle/input/<slug>/ and only scores
# files written to /kaggle/working. Locally, drop the CSVs into ./data/ or
# point DSDG_DATA_DIR at them.
# DSDG_KAGGLE_ROOT exists so the Kaggle branch below is testable off Kaggle;
# unset, it is the real thing.
KAGGLE_ROOT = Path(os.environ.get("DSDG_KAGGLE_ROOT", "/kaggle"))
KAGGLE_INPUT = KAGGLE_ROOT / "input"
KAGGLE_WORKING = KAGGLE_ROOT / "working"
ON_KAGGLE = KAGGLE_WORKING.is_dir()

# Writing anywhere else on Kaggle produces a notebook that runs green and
# submits nothing, so the output path follows the environment rather than
# needing an edit between the two.
OUT_DIR = KAGGLE_WORKING if ON_KAGGLE else PROJECT_DIR / "out"
LOCAL_DATA = PROJECT_DIR / "data"
DATA_DIR_ENV = "DSDG_DATA_DIR"

# Leaderboard split, from the cheat sheet. The horizon itself is read off
# sample_submission.csv rather than hardcoded -- see schema.py.
PUBLIC_WEEKS = range(209, 222)
PRIVATE_WEEKS = range(222, 235)

# Phase 1 data covers weeks 1-208; phase 2 extends it to 221. Detected at load
# time; these are only the expected values, used for a sanity warning.
PHASE1_LAST_WEEK = 208
PHASE2_LAST_WEEK = 221


@dataclass
class CostModel:
    """Cost constants the local simulator charges.

    Only `stockout_multiple` is stated on the cheat sheet ("a stockout costs 5
    times what the material does"). The rest are guesses with the right sign;
    fix them from the competition description before you trust a local score.
    """

    stockout_multiple: float = 5.0          # known: 5x the material's cost
    holding_cost_per_unit_week: float = 0.01  # CONFIRM: fraction of unit price
    lead_time_weeks: int = 1                 # CONFIRM: order week -> arrival week
    initial_inventory_units: float = 0.0     # CONFIRM: what you start week 209 with

    # Weeks of cover the policy aims to hold on top of expected demand. Tuned
    # on weeks 209-221 once phase 2 lands, not guessed.
    target_weeks_of_cover: float = 1.0

    confirmed: set[str] = field(default_factory=set)

    CONFIRMABLE = (
        "holding_cost_per_unit_week",
        "lead_time_weeks",
        "initial_inventory_units",
    )

    @property
    def critical_ratio(self) -> float:
        """Newsvendor service level implied by the cost asymmetry.

        Short one unit costs `stockout_multiple` x price. Long one unit costs a
        week of holding. The optimal order quantity covers demand up to this
        quantile, which is why over-ordering is correct here.
        """
        underage = self.stockout_multiple
        overage = self.holding_cost_per_unit_week
        return underage / (underage + overage)

    def audit(self) -> list[str]:
        """Constants still unconfirmed against the competition docs."""
        return [name for name in self.CONFIRMABLE if name not in self.confirmed]

    def confirm(self, *names: str) -> "CostModel":
        for name in names:
            if name not in self.CONFIRMABLE:
                raise KeyError(f"{name!r} is not a confirmable constant")
            self.confirmed.add(name)
        return self


COSTS = CostModel()


def data_dir() -> Path:
    """Locate the competition data, preferring an explicit override."""
    override = os.environ.get(DATA_DIR_ENV)
    if override:
        path = Path(override)
        if not (path / "sample_submission.csv").exists():
            raise FileNotFoundError(
                f"{DATA_DIR_ENV}={path} has no sample_submission.csv"
            )
        return path

    if KAGGLE_INPUT.is_dir():
        for candidate in sorted(KAGGLE_INPUT.iterdir()):
            if (candidate / "sample_submission.csv").exists():
                return candidate

    if (LOCAL_DATA / "sample_submission.csv").exists():
        return LOCAL_DATA

    # The competition download unpacks into its own folder at the project root.
    for candidate in sorted(PROJECT_DIR.iterdir()):
        if candidate.is_dir() and (candidate / "sample_submission.csv").exists():
            return candidate

    raise FileNotFoundError(
        "No sample_submission.csv found. Looked at "
        f"{KAGGLE_INPUT}/*/, {LOCAL_DATA}, and ${DATA_DIR_ENV}."
    )


In [ ]:
%%writefile schema.py
"""Submission schema, read off sample_submission.csv instead of hardcoded.

The cheat sheet gives 260 rows and share columns "share_KES to share_AUR", but
not the material list, the country list, or their order. sample_submission.csv
has all three and is guaranteed valid, so it is the source of truth: parse it
once and every other module derives from the result. Nothing downstream needs
editing if a code turns out to be spelled differently than expected.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import re

import pandas as pd

import config

ID_COL = "id"
PROB_COL = "disruption_prob"
PRICE_COL = "price_forecast"
ORDER_COL = "order_units"
SHARE_PREFIX = "share_"

VALUE_COLS = (PROB_COL, PRICE_COL, ORDER_COL)
ID_PATTERN = re.compile(r"^w(?P<week>\d+)_(?P<material>.+)$")


@dataclass(frozen=True)
class Schema:
    """Everything a valid submission must line up with."""

    ids: tuple[str, ...]
    weeks: tuple[int, ...]
    materials: tuple[str, ...]
    countries: tuple[str, ...]
    columns: tuple[str, ...]

    @property
    def share_cols(self) -> tuple[str, ...]:
        return tuple(f"{SHARE_PREFIX}{c}" for c in self.countries)

    @property
    def n_rows(self) -> int:
        return len(self.ids)

    def make_id(self, week: int, material: str) -> str:
        return f"w{week}_{material}"

    def index(self) -> pd.MultiIndex:
        """(week, material) index in the submission's own row order."""
        parsed = [parse_id(i) for i in self.ids]
        return pd.MultiIndex.from_tuples(parsed, names=["week", "material"])

    def describe(self) -> str:
        return (
            f"{self.n_rows} rows = {len(self.weeks)} weeks "
            f"(w{self.weeks[0]}-w{self.weeks[-1]}) x {len(self.materials)} materials, "
            f"{len(self.countries)} countries: {', '.join(self.countries)}"
        )


def parse_id(row_id: str) -> tuple[int, str]:
    match = ID_PATTERN.match(row_id)
    if not match:
        raise ValueError(f"id {row_id!r} is not of the form w{{week}}_{{material}}")
    return int(match["week"]), match["material"]


def load(sample_path: Path | None = None) -> Schema:
    """Read the schema out of sample_submission.csv."""
    path = sample_path or (config.data_dir() / "sample_submission.csv")
    sample = pd.read_csv(path)

    missing = [c for c in (ID_COL, *VALUE_COLS) if c not in sample.columns]
    if missing:
        raise ValueError(f"{path.name} is missing expected columns: {missing}")

    share_cols = [c for c in sample.columns if c.startswith(SHARE_PREFIX)]
    if not share_cols:
        raise ValueError(f"{path.name} has no {SHARE_PREFIX}* columns")

    parsed = [parse_id(i) for i in sample[ID_COL]]
    weeks = sorted({week for week, _ in parsed})
    # Keep first-seen material order so output ordering is stable and readable.
    materials = list(dict.fromkeys(material for _, material in parsed))

    return Schema(
        ids=tuple(sample[ID_COL]),
        weeks=tuple(weeks),
        materials=tuple(materials),
        countries=tuple(c[len(SHARE_PREFIX):] for c in share_cols),
        columns=tuple(sample.columns),
    )


def eligibility(schema: Schema, suppliers: pd.DataFrame | None) -> pd.DataFrame:
    """material x country mask of who actually sells what.

    A country that doesn't sell a material must get share 0, so this mask gates
    every allocation. It is read from the supplier table when one is available;
    otherwise everything is marked eligible and the caller is warned, because
    guessing the wrong zeros silently breaks the resilience score.
    """
    mask = pd.DataFrame(
        True, index=list(schema.materials), columns=list(schema.countries)
    )
    if suppliers is None or suppliers.empty:
        mask.attrs["inferred"] = False
        return mask

    material_col = _find_col(suppliers, ("material", "material_name", "commodity"))
    country_col = _find_col(suppliers, ("country", "country_code", "supplier_country"))
    if material_col is None or country_col is None:
        mask.attrs["inferred"] = False
        return mask

    mask.loc[:, :] = False
    for material, country in zip(suppliers[material_col], suppliers[country_col]):
        if material in mask.index and country in mask.columns:
            mask.loc[material, country] = True

    # A material nobody appears to supply is a parse failure, not a real fact.
    orphans = mask.index[~mask.any(axis=1)]
    if len(orphans):
        mask.loc[orphans, :] = True
    mask.attrs["inferred"] = True
    mask.attrs["orphans"] = list(orphans)
    return mask


def _find_col(frame: pd.DataFrame, candidates: tuple[str, ...]) -> str | None:
    lowered = {c.lower(): c for c in frame.columns}
    for candidate in candidates:
        if candidate in lowered:
            return lowered[candidate]
    return None


In [ ]:
%%writefile data.py
"""Load whatever CSVs the competition ships, without naming them in advance.

Phase 2 at Saturday noon replaces the same filenames with more weeks, so this
discovers files and reports the week coverage it found rather than assuming
1-208. `Dataset.last_week` is what tells you the rerun actually picked up the
new data.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path

import pandas as pd

import config
import schema as schema_mod

WEEK_COL_CANDIDATES = ("week", "week_num", "week_index", "t")
SUPPLIER_HINTS = ("supplier", "sourcing", "vendor")
PRICE_HINTS = ("price", "spot")
DISRUPTION_HINTS = ("disrupt", "shock", "event", "outage")
DEMAND_HINTS = ("demand", "consumption", "usage")


@dataclass
class Dataset:
    """Every table found, plus the schema and a few convenience views."""

    dir: Path
    tables: dict[str, pd.DataFrame]
    schema: schema_mod.Schema
    sample: pd.DataFrame
    eligibility: pd.DataFrame = field(init=False)

    def __post_init__(self) -> None:
        self.eligibility = schema_mod.eligibility(self.schema, self.suppliers)

    def table(self, *hints: str) -> pd.DataFrame | None:
        """First table whose filename matches any hint."""
        for name, frame in self.tables.items():
            if any(hint in name.lower() for hint in hints):
                return frame
        return None

    @property
    def suppliers(self) -> pd.DataFrame | None:
        return self.table(*SUPPLIER_HINTS)

    @property
    def prices(self) -> pd.DataFrame | None:
        return self.table(*PRICE_HINTS)

    @property
    def disruptions(self) -> pd.DataFrame | None:
        return self.table(*DISRUPTION_HINTS)

    @property
    def demand(self) -> pd.DataFrame | None:
        return self.table(*DEMAND_HINTS)

    @property
    def last_week(self) -> int | None:
        """Highest week number observed in any table -- 208 in phase 1, 221 in phase 2."""
        seen = [
            int(frame[col].max())
            for frame in self.tables.values()
            for col in (week_col(frame),)
            if col is not None and len(frame)
        ]
        return max(seen) if seen else None

    @property
    def phase(self) -> str:
        last = self.last_week
        if last is None:
            return "unknown"
        if last >= config.PHASE2_LAST_WEEK:
            return "phase2"
        if last >= config.PHASE1_LAST_WEEK:
            return "phase1"
        return f"partial (through w{last})"

    def summary(self) -> str:
        lines = [
            f"data dir: {self.dir}",
            f"schema:   {self.schema.describe()}",
            f"coverage: {self.phase} (last observed week {self.last_week})",
        ]
        for name, frame in sorted(self.tables.items()):
            cols = ", ".join(frame.columns[:8])
            more = " ..." if len(frame.columns) > 8 else ""
            lines.append(f"  {name:<28} {frame.shape[0]:>7} x {frame.shape[1]:<3} [{cols}{more}]")
        if not self.eligibility.attrs.get("inferred"):
            lines.append(
                "  WARNING: no supplier table parsed -- every country is marked "
                "eligible for every material. Fix before trusting shares."
            )
        return "\n".join(lines)


def week_col(frame: pd.DataFrame) -> str | None:
    lowered = {c.lower(): c for c in frame.columns}
    for candidate in WEEK_COL_CANDIDATES:
        if candidate in lowered:
            return lowered[candidate]
    return None


def load(data_path: Path | None = None) -> Dataset:
    """Read every CSV in the data directory."""
    directory = data_path or config.data_dir()
    tables = {
        path.name: pd.read_csv(path)
        for path in sorted(directory.glob("*.csv"))
        if path.name != "sample_submission.csv"
    }
    sample = pd.read_csv(directory / "sample_submission.csv")
    return Dataset(
        dir=directory,
        tables=tables,
        schema=schema_mod.load(directory / "sample_submission.csv"),
        sample=sample,
    )


if __name__ == "__main__":
    print(load().summary())


In [ ]:
%%writefile forecast.py
"""Baseline forecasts, and the column-finding needed to build them.

Replace these with the real model -- they exist so that a valid, non-trivial
submission is reachable within minutes of the data dropping, and so there is
something to beat. Each returns a Series indexed by (week, material), which is
the shape every other module expects.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

MATERIAL_COL_CANDIDATES = ("material", "material_name", "commodity", "item")
COUNTRY_COL_CANDIDATES = ("country", "country_code", "supplier_country")
SEASON_PERIOD = 52  # weeks; the data is weekly, so a year is the obvious cycle


def find_col(frame: pd.DataFrame, candidates: tuple[str, ...]) -> str | None:
    lowered = {c.lower(): c for c in frame.columns}
    for candidate in candidates:
        if candidate in lowered:
            return lowered[candidate]
    for key, original in lowered.items():
        if any(candidate in key for candidate in candidates):
            return original
    return None


def to_panel(
    frame: pd.DataFrame, value_col: str, week_col: str, material_col: str
) -> pd.Series:
    """Collapse a table to one value per (week, material)."""
    grouped = frame.groupby([week_col, material_col])[value_col].mean()
    grouped.index.names = ["week", "material"]
    return grouped.sort_index()


def to_country_panel(
    frame: pd.DataFrame, value_col: str, week_col: str, material_col: str, country_col: str
) -> pd.DataFrame:
    """(week, material) x country frame, for per-supplier prices or risk."""
    wide = frame.pivot_table(
        index=[week_col, material_col], columns=country_col, values=value_col, aggfunc="mean"
    )
    wide.index.names = ["week", "material"]
    return wide.sort_index()


def seasonal_naive(
    history: pd.Series, horizon: list[int], period: int = SEASON_PERIOD, blend: float = 0.5
) -> pd.Series:
    """Blend of "same week last year" and the recent mean, per material.

    Seasonal-naive alone is noisy on a single draw; the recent mean alone
    ignores any annual cycle. Blending both is a sturdier baseline than either
    and takes one line to beat with a real model.
    """
    out = {}
    recent_window = min(13, max(1, len(history.index.get_level_values("week").unique())))

    for material, series in history.groupby(level="material"):
        by_week = series.droplevel("material")
        recent = by_week.tail(recent_window).mean()
        for week in horizon:
            prior = by_week.get(week - period, np.nan)
            if np.isnan(prior):
                value = recent
            else:
                value = blend * prior + (1.0 - blend) * recent
            out[(week, material)] = float(value)

    return _as_panel(out)


def dispersion(history: pd.Series, horizon: list[int], window: int = 52) -> pd.Series:
    """Per-material standard deviation, carried flat across the horizon.

    Feeds the safety-stock term in the ordering policy, so a material with
    volatile demand is held deeper than a steady one.
    """
    out = {}
    for material, series in history.groupby(level="material"):
        sd = float(series.droplevel("material").tail(window).std(ddof=0) or 0.0)
        for week in horizon:
            out[(week, material)] = sd
    return _as_panel(out)


def base_rate(events: pd.Series, horizon: list[int], prior_weight: float = 10.0) -> pd.Series:
    """Historical disruption frequency per material, smoothed toward the pooled rate.

    A material with few observed shocks should not be forecast at 0.0 -- the
    5x stockout penalty makes that the expensive direction to be wrong in.
    """
    pooled = float(events.mean())
    out = {}
    for material, series in events.groupby(level="material"):
        hits, n = float(series.sum()), float(len(series))
        rate = (hits + prior_weight * pooled) / (n + prior_weight)
        for week in horizon:
            out[(week, material)] = min(max(rate, 0.0), 1.0)
    return _as_panel(out)


def flat(value: float, horizon: list[int], materials: list[str]) -> pd.Series:
    """Constant fallback, for when a table could not be located at all."""
    return _as_panel({(w, m): float(value) for w in horizon for m in materials})


def _as_panel(mapping: dict[tuple[int, str], float]) -> pd.Series:
    series = pd.Series(mapping, dtype=float)
    series.index = pd.MultiIndex.from_tuples(series.index, names=["week", "material"])
    return series.sort_index()


def carry_forward_country(
    panel: pd.DataFrame, horizon: list[int], window: int = 13
) -> pd.DataFrame:
    """Project a (week, material) x country history onto the horizon.

    The per-supplier price and risk frames are history; the share allocator
    needs them on the forecast weeks. Reindexing history straight onto the
    horizon yields all-NaN and silently disables the tilt, so each material
    carries its recent per-country mean forward instead.
    """
    out = {}
    for material, block in panel.groupby(level="material"):
        recent = block.droplevel("material").tail(window).mean()
        for week in horizon:
            out[(week, material)] = recent

    frame = pd.DataFrame.from_dict(out, orient="index")
    frame.index = pd.MultiIndex.from_tuples(frame.index, names=["week", "material"])
    return frame.sort_index()


In [ ]:
%%writefile policy.py
"""Turn forecasts into orders and sourcing shares.

This is the half of the submission the simulator actually plays, and it carries
more weight than the forecasts do, so it is deliberately explicit rather than
learned. Two ideas do the work:

1. The cost asymmetry. A stockout costs 5x the material, so the newsvendor
   service level is config.COSTS.critical_ratio -- far above 50%. Carrying
   extra is cheap; running out is not.
2. Buy early and spread wide ahead of a shock. A disruption forecast raises
   both the target stock for the weeks before it and the weight on suppliers
   that are not expected to be hit.
"""

from __future__ import annotations

from statistics import NormalDist

import numpy as np
import pandas as pd

import config
from config import CostModel

_NORMAL = NormalDist()


def safety_factor(costs: CostModel) -> float:
    """z for the service level implied by the cost ratio."""
    return _NORMAL.inv_cdf(min(costs.critical_ratio, 0.999))


def base_stock_orders(
    demand_mean: pd.Series,
    demand_sd: pd.Series,
    disruption_prob: pd.Series,
    costs: CostModel | None = None,
    inventory_start: float | dict[str, float] | None = None,
) -> pd.DataFrame:
    """Order quantities from a base-stock policy, rolled forward week by week.

    All three inputs are indexed by (week, material). Returns a frame with the
    order plus the projected inventory and target that produced it, so you can
    see why a week ordered what it did.

    The target covers expected demand over the lead time plus the configured
    weeks of cover, a safety buffer scaled by the service level, and -- when a
    disruption is forecast inside the arrival window -- the demand that a lost
    delivery would leave uncovered. That last term is the "buy early" move: it
    lands stock *before* the shock week, not during it.
    """
    costs = costs or config.COSTS
    z = safety_factor(costs)
    horizon = costs.lead_time_weeks + costs.target_weeks_of_cover

    index = demand_mean.index
    materials = list(dict.fromkeys(index.get_level_values("material")))
    weeks = sorted(set(index.get_level_values("week")))

    if isinstance(inventory_start, dict):
        inventory = dict(inventory_start)
    else:
        default = costs.initial_inventory_units if inventory_start is None else inventory_start
        inventory = {m: float(default) for m in materials}

    pipeline: dict[str, dict[int, float]] = {m: {} for m in materials}
    rows = []

    for week in weeks:
        for material in materials:
            key = (week, material)
            if key not in demand_mean.index:
                continue

            inventory[material] += pipeline[material].pop(week, 0.0)
            inventory[material] = max(0.0, inventory[material] - float(demand_mean[key]))

            mean = float(demand_mean[key])
            sd = float(demand_sd.get(key, 0.0))
            risk = float(disruption_prob.get(key, 0.0))

            in_transit = sum(pipeline[material].values())
            target = mean * horizon + z * sd * np.sqrt(max(horizon, 1.0))
            # Expected shortfall if the incoming delivery is disrupted.
            target += risk * mean * costs.lead_time_weeks

            order = max(0.0, target - inventory[material] - in_transit)
            arrival = week + costs.lead_time_weeks
            pipeline[material][arrival] = pipeline[material].get(arrival, 0.0) + order

            rows.append(
                {
                    "week": week,
                    "material": material,
                    "order_units": order,
                    "projected_inventory": inventory[material],
                    "target": target,
                    "in_transit": in_transit,
                }
            )

    return pd.DataFrame(rows).set_index(["week", "material"])


def risk_tilted_shares(
    schema_countries: list[str],
    eligibility: pd.DataFrame,
    index: pd.MultiIndex,
    country_risk: pd.DataFrame | None = None,
    country_price: pd.DataFrame | None = None,
    diversification: float = 0.5,
    sharpness: float = 3.0,
    min_suppliers: int = 2,
) -> pd.DataFrame:
    """Allocate each order across countries, indexed by (week, material).

    Weights are inverse-risk and inverse-price among eligible suppliers, then
    blended toward an even split by `diversification` (0 = chase the single
    cheapest safe supplier, 1 = always split evenly). The blend is the lever
    the resilience score responds to: concentrating on one supplier scores well
    on cost right up until that supplier is the one that gets hit.

    `sharpness` is what gives that lever range. Raw inverse-price weights are
    nearly flat whenever suppliers are similarly priced, so the tilt is raised
    to this power first: 1.0 is the gentle version, higher values push toward
    winner-take-all. Tune the pair together against weeks 209-221.

    `country_risk` and `country_price` are optional (week, material) x country
    frames. Without them every eligible supplier is weighted equally, which is
    a valid, middling-scoring baseline.
    """
    if not 0.0 <= diversification <= 1.0:
        raise ValueError("diversification must be in [0, 1]")

    materials = index.get_level_values("material")
    mask = eligibility.reindex(materials.values)[schema_countries].to_numpy().astype(float)

    weights = np.ones_like(mask)
    if country_risk is not None:
        risk = country_risk.reindex(index=index, columns=schema_countries).fillna(0.0)
        weights *= (1.0 - risk.to_numpy().clip(0.0, 0.999))
    if country_price is not None:
        price = country_price.reindex(index=index, columns=schema_countries)
        price = price.where(price > 0).to_numpy()
        with np.errstate(invalid="ignore", divide="ignore"):
            inverse = np.where(np.isnan(price), 1.0, 1.0 / price)
        weights *= inverse

    weights = np.where(np.isfinite(weights), weights, 0.0)
    # Normalise each row before exponentiating so `sharpness` acts on the
    # relative gaps between suppliers, not on the absolute price scale.
    scale = weights.max(axis=1, keepdims=True)
    weights = np.divide(weights, scale, out=np.zeros_like(weights), where=scale > 0)
    weights = weights ** max(sharpness, 0.0)
    weights *= mask

    totals = weights.sum(axis=1, keepdims=True)
    even = mask / np.maximum(mask.sum(axis=1, keepdims=True), 1.0)
    tilted = np.divide(weights, totals, out=even.copy(), where=totals > 0)

    blended = (1.0 - diversification) * tilted + diversification * even
    blended = _spread_to_min_suppliers(blended, mask, min_suppliers)

    totals = blended.sum(axis=1, keepdims=True)
    blended = np.divide(blended, totals, out=even.copy(), where=totals > 0)
    return pd.DataFrame(blended, index=index, columns=schema_countries)


def _spread_to_min_suppliers(
    weights: np.ndarray, mask: np.ndarray, min_suppliers: int
) -> np.ndarray:
    """Guarantee a floor on how many eligible suppliers get a real share.

    A single-sourced row is one disruption away from a stockout, and a stockout
    costs 5x. Rows with fewer than `min_suppliers` non-trivial shares get a
    small floor spread across their eligible suppliers.
    """
    if min_suppliers <= 1:
        return weights

    floor = 0.05
    live = (weights > floor / 2).sum(axis=1)
    available = mask.sum(axis=1).astype(int)
    needs_help = (live < np.minimum(min_suppliers, available)) & (available > 1)
    if not needs_help.any():
        return weights

    out = weights.copy()
    out[needs_help] = np.maximum(out[needs_help], floor * mask[needs_help])
    return out


In [ ]:
%%writefile simulator.py
"""Local scorer: play a submission's orders against what really happened.

This is the file worth the most. Five submissions a day is not enough to tune
a policy by probing the leaderboard, so the policy gets tuned here instead and
the leaderboard only confirms. Once phase 2 lands on Saturday noon, weeks
209-221 have known outcomes and become the validation set -- same simulator,
one week past the training cutoff, the closest thing to the private weeks.

The mechanics below are a reasonable model, not the official one:

  * an order placed in week w arrives in w + lead_time
  * a disrupted supplier delivers a REDUCED fraction of what was ordered from
    it, not zero -- the EDA found capacity ratios averaging 0.37 on hit weeks
    with a floor of 0.05, never 0, so treating a shock as a failed delivery
    overstates stockout risk and makes the policy over-order
  * unmet demand is a stockout, charged at stockout_multiple x benchmark price
  * leftover stock carries to the next week at holding_cost_per_unit_week

Check each against the competition description and call `costs.confirm(...)` as
you verify them. `score()` refuses to pretend it is the real scorer while
constants are unconfirmed -- it reports components and says so.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd

import config
from config import CostModel


@dataclass
class Outcome:
    """Per (week, material) ledger plus the aggregates built from it."""

    ledger: pd.DataFrame
    costs: CostModel

    @property
    def total_cost(self) -> float:
        return float(self.ledger[["purchase_cost", "holding_cost", "stockout_cost"]].sum().sum())

    @property
    def stockout_units(self) -> float:
        return float(self.ledger["stockout_units"].sum())

    @property
    def stockout_weeks(self) -> int:
        return int((self.ledger["stockout_units"] > 0).sum())

    @property
    def service_level(self) -> float:
        demand = float(self.ledger["demand"].sum())
        return 1.0 - self.stockout_units / demand if demand else 1.0

    def summary(self) -> str:
        return (
            f"total cost     {self.total_cost:,.0f}\n"
            f"  purchase     {self.ledger['purchase_cost'].sum():,.0f}\n"
            f"  holding      {self.ledger['holding_cost'].sum():,.0f}\n"
            f"  stockout     {self.ledger['stockout_cost'].sum():,.0f}\n"
            f"service level  {self.service_level:.3%}\n"
            f"stockout weeks {self.stockout_weeks} of {len(self.ledger)}"
        )


def rollout(
    orders: pd.Series,
    shares: pd.DataFrame,
    demand: pd.Series,
    prices: pd.Series,
    country_capacity: pd.DataFrame,
    costs: CostModel | None = None,
) -> Outcome:
    """Simulate the horizon week by week.

    orders, demand, prices  -- indexed by (week, material)
    shares                  -- (week, material) x country, rows summing to 1
    country_capacity        -- (week, material) x country, the fraction of what
                               is ordered from that supplier that actually
                               ships: 1.0 when healthy, ~0.37 on average when
                               hit, never 0. A boolean frame of disruption
                               flags also works (True -> 0.0), but that is the
                               pessimistic reading the EDA ruled out.
    """
    costs = costs or config.COSTS
    index = orders.index
    materials = list(dict.fromkeys(index.get_level_values("material")))
    weeks = sorted(set(index.get_level_values("week")))

    inventory = {m: float(costs.initial_inventory_units) for m in materials}
    arrivals: dict[str, dict[int, float]] = {m: {} for m in materials}
    rows = []

    for week in weeks:
        for material in materials:
            key = (week, material)
            if key not in index:
                continue

            received = arrivals[material].pop(week, 0.0)
            inventory[material] += received

            want = float(demand.get(key, 0.0))
            used = min(inventory[material], want)
            inventory[material] -= used
            short = want - used

            ordered = float(orders[key])
            price = float(prices.get(key, 0.0))
            row_shares = shares.loc[key] if key in shares.index else None
            cap = (
                country_capacity.loc[key]
                if country_capacity is not None and key in country_capacity.index
                else None
            )

            # Expected delivered fraction, weighted by where the order was placed.
            arrive_fraction = 1.0
            if row_shares is not None and cap is not None:
                aligned = cap.reindex(row_shares.index)
                if aligned.dtype == bool:
                    aligned = (~aligned).astype(float)
                else:
                    aligned = aligned.fillna(1.0).clip(0.0, 1.0)
                arrive_fraction = float((row_shares * aligned).sum())
            lost_fraction = 1.0 - arrive_fraction

            will_arrive = ordered * arrive_fraction
            arrival_week = week + costs.lead_time_weeks
            arrivals[material][arrival_week] = (
                arrivals[material].get(arrival_week, 0.0) + will_arrive
            )

            rows.append(
                {
                    "week": week,
                    "material": material,
                    "demand": want,
                    "received": received,
                    "ordered": ordered,
                    "lost_fraction": lost_fraction,
                    "ending_inventory": inventory[material],
                    "stockout_units": short,
                    "purchase_cost": ordered * price,
                    "holding_cost": inventory[material] * price * costs.holding_cost_per_unit_week,
                    "stockout_cost": short * price * costs.stockout_multiple,
                }
            )

    return Outcome(pd.DataFrame(rows).set_index(["week", "material"]), costs)


# --- prediction metrics -----------------------------------------------------

def brier(prob: pd.Series, actual: pd.Series) -> float:
    """Mean squared error of the disruption probabilities. Lower is better."""
    p, a = _align(prob, actual)
    return float(np.mean((p - a) ** 2))


def auc(prob: pd.Series, actual: pd.Series) -> float:
    """Rank-based AUC, computed without sklearn so it runs anywhere."""
    p, a = _align(prob, actual)
    positives, negatives = p[a > 0.5], p[a <= 0.5]
    if not len(positives) or not len(negatives):
        return float("nan")
    ranks = pd.Series(np.concatenate([positives, negatives])).rank().to_numpy()
    rank_sum = ranks[: len(positives)].sum()
    return float(
        (rank_sum - len(positives) * (len(positives) + 1) / 2)
        / (len(positives) * len(negatives))
    )


def price_error(forecast: pd.Series, actual: pd.Series) -> dict[str, float]:
    f, a = _align(forecast, actual)
    with np.errstate(divide="ignore", invalid="ignore"):
        mape = np.nanmean(np.abs((f - a) / np.where(a == 0, np.nan, a)))
    return {
        "rmse": float(np.sqrt(np.mean((f - a) ** 2))),
        "mae": float(np.mean(np.abs(f - a))),
        "mape": float(mape),
    }


def _align(left: pd.Series, right: pd.Series) -> tuple[np.ndarray, np.ndarray]:
    shared = left.index.intersection(right.index)
    if not len(shared):
        raise ValueError("forecast and actual share no (week, material) keys")
    return left.reindex(shared).to_numpy(float), right.reindex(shared).to_numpy(float)


# --- resilience proxies -----------------------------------------------------

def concentration(shares: pd.DataFrame) -> pd.Series:
    """Herfindahl index per row: 1.0 is single-sourced, 1/n is an even split."""
    return (shares ** 2).sum(axis=1)


def resilience_proxies(shares: pd.DataFrame, outcome: Outcome) -> dict[str, float]:
    """Stand-ins for the real resilience component, whose formula is not public.

    Useful for ranking two of your own policies against each other; not
    comparable to the leaderboard's 0-100 scale.
    """
    hhi = concentration(shares)
    return {
        "mean_hhi": float(hhi.mean()),
        "max_hhi": float(hhi.max()),
        "single_sourced_rows": int((hhi > 0.99).sum()),
        "mean_suppliers_used": float((shares > 0.01).sum(axis=1).mean()),
        "service_level": outcome.service_level,
        "stockout_weeks": outcome.stockout_weeks,
    }


def report(
    outcome: Outcome,
    shares: pd.DataFrame,
    prob: pd.Series | None = None,
    actual_disruption: pd.Series | None = None,
    price_forecast: pd.Series | None = None,
    actual_price: pd.Series | None = None,
) -> str:
    """Everything the local scorer knows, with its caveat attached."""
    lines = ["=== financial ===", outcome.summary(), "", "=== resilience (proxy) ==="]
    for name, value in resilience_proxies(shares, outcome).items():
        lines.append(f"{name:<22} {value:,.4f}")

    if prob is not None and actual_disruption is not None:
        lines += [
            "",
            "=== prediction ===",
            f"{'brier':<22} {brier(prob, actual_disruption):.4f}",
            f"{'auc':<22} {auc(prob, actual_disruption):.4f}",
        ]
    if price_forecast is not None and actual_price is not None:
        for name, value in price_error(price_forecast, actual_price).items():
            lines.append(f"{'price_' + name:<22} {value:,.4f}")

    if unconfirmed := outcome.costs.audit():
        lines += [
            "",
            "NOTE: these are relative numbers only -- the following constants "
            "are still placeholders: " + ", ".join(unconfirmed),
        ]
    return "\n".join(lines)


In [ ]:
%%writefile submission.py
"""Build and validate the submission CSV.

Every rule the cheat sheet calls a trip-up is an assertion here, so a bad file
fails locally in a second instead of burning one of five daily submissions.
Shares are renormalised to sum to exactly 1 after the eligibility mask is
applied -- doing it in the other order is what silently produces rows summing
to 0.98.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

import config
from schema import (
    ID_COL,
    ORDER_COL,
    PRICE_COL,
    PROB_COL,
    SHARE_PREFIX,
    Schema,
)

SHARE_TOLERANCE = 1e-6
MIN_PRICE = 1e-9  # price_forecast must be strictly above 0


@dataclass
class Report:
    errors: list[str]
    warnings: list[str]

    @property
    def ok(self) -> bool:
        return not self.errors

    def __str__(self) -> str:
        if self.ok and not self.warnings:
            return "submission valid"
        lines = [f"ERROR: {e}" for e in self.errors]
        lines += [f"warn:  {w}" for w in self.warnings]
        lines.append("submission valid" if self.ok else f"{len(self.errors)} error(s)")
        return "\n".join(lines)

    def raise_if_bad(self) -> None:
        if not self.ok:
            raise ValueError("invalid submission:\n" + str(self))


def normalise_shares(
    shares: pd.DataFrame, eligibility: pd.DataFrame, materials: pd.Series
) -> pd.DataFrame:
    """Zero out ineligible countries, then make each row sum to exactly 1.

    A row left with no eligible supplier would divide by zero; it falls back to
    an even split across every country, which is wrong but valid, and the
    validator's eligibility warning will point at it.
    """
    shares = shares.clip(lower=0.0)
    mask = eligibility.reindex(materials.values).to_numpy()
    values = np.where(mask, shares.to_numpy(), 0.0)

    totals = values.sum(axis=1, keepdims=True)
    dead = totals.squeeze(axis=1) <= 0
    if dead.any():
        values[dead] = 1.0 / values.shape[1]
        totals = values.sum(axis=1, keepdims=True)

    return pd.DataFrame(values / totals, index=shares.index, columns=shares.columns)


def build(
    schema: Schema,
    prob: pd.Series,
    price: pd.Series,
    orders: pd.Series,
    shares: pd.DataFrame,
    eligibility: pd.DataFrame,
) -> pd.DataFrame:
    """Assemble the submission frame in the sample's own row and column order.

    `prob`, `price`, `orders` and `shares` are indexed by (week, material);
    reindexing against the schema is what guarantees all rows are present.
    """
    index = schema.index()
    frame = pd.DataFrame(index=index)
    frame[PROB_COL] = prob.reindex(index).clip(0.0, 1.0)
    frame[PRICE_COL] = price.reindex(index).clip(lower=MIN_PRICE)
    frame[ORDER_COL] = orders.reindex(index).clip(lower=0.0)

    materials = index.get_level_values("material").to_series(index=index)
    aligned = shares.reindex(index=index, columns=list(schema.countries)).fillna(0.0)
    normalised = normalise_shares(aligned, eligibility, materials)
    for country in schema.countries:
        frame[f"{SHARE_PREFIX}{country}"] = normalised[country]

    frame.insert(0, ID_COL, [schema.make_id(w, m) for w, m in index])
    return frame.reset_index(drop=True)[list(schema.columns)]


def validate(
    frame: pd.DataFrame, schema: Schema, eligibility: pd.DataFrame | None = None
) -> Report:
    errors: list[str] = []
    warnings: list[str] = []

    if list(frame.columns) != list(schema.columns):
        errors.append(
            f"columns differ from sample_submission\n"
            f"       got:      {list(frame.columns)}\n"
            f"       expected: {list(schema.columns)}"
        )
        return Report(errors, warnings)

    if len(frame) != schema.n_rows:
        errors.append(f"{len(frame)} rows, expected {schema.n_rows}")

    got, expected = set(frame[ID_COL]), set(schema.ids)
    if missing := sorted(expected - got):
        errors.append(f"{len(missing)} missing id(s), e.g. {missing[:5]}")
    if extra := sorted(got - expected):
        errors.append(f"{len(extra)} unexpected id(s), e.g. {extra[:5]}")
    if len(got) != len(frame):
        errors.append("duplicate ids")

    numeric = frame.drop(columns=[ID_COL])
    if nan_cols := sorted(numeric.columns[numeric.isna().any()]):
        errors.append(f"NaN values in {nan_cols}")

    probs = frame[PROB_COL]
    if ((probs < 0) | (probs > 1)).any():
        errors.append(f"{int(((probs < 0) | (probs > 1)).sum())} disruption_prob outside [0, 1]")

    if (frame[PRICE_COL] <= 0).any():
        errors.append(f"{int((frame[PRICE_COL] <= 0).sum())} price_forecast not above 0")

    if (frame[ORDER_COL] < 0).any():
        errors.append(f"{int((frame[ORDER_COL] < 0).sum())} negative order_units")

    shares = frame[list(schema.share_cols)]
    if (shares < 0).any().any():
        errors.append("negative share values")
    off = (shares.sum(axis=1) - 1.0).abs()
    if (off > SHARE_TOLERANCE).any():
        worst = off.max()
        errors.append(
            f"{int((off > SHARE_TOLERANCE).sum())} row(s) whose shares do not sum to 1 "
            f"(worst off by {worst:.2e})"
        )

    if eligibility is not None:
        materials = frame[ID_COL].str.split("_", n=1).str[1]
        mask = eligibility.reindex(materials.values).to_numpy()
        violations = int(((shares.to_numpy() > 0) & ~mask).sum())
        if violations:
            errors.append(f"{violations} non-zero share(s) for a country that does not sell the material")
        if not eligibility.attrs.get("inferred", False):
            warnings.append(
                "eligibility was not read from a supplier table, so the "
                "material/country zeros above are unchecked"
            )

    if unconfirmed := config.COSTS.audit():
        warnings.append(f"unconfirmed cost constants: {', '.join(unconfirmed)}")

    return Report(errors, warnings)


def write(frame: pd.DataFrame, schema: Schema, path: Path, eligibility=None) -> Path:
    """Validate, then write. Never writes a file that would score as invalid."""
    validate(frame, schema, eligibility).raise_if_bad()
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    return path


In [ ]:
%%writefile run.py
"""End-to-end baseline: data in, valid submission out.

    python run.py                      # write out/submission.csv
    python run.py --diversification 0.7 --cover 2.0
    python run.py --dry-run            # build and validate, write nothing

It is deliberately tolerant about column names: it reports what it located and
what it had to fall back on, so the first run against real data tells you
exactly which three or four lines to wire up. A FALLBACK on demand or price
means the orders are placeholders, not a strategy.
"""

from __future__ import annotations

import argparse
from pathlib import Path

import pandas as pd

import config
import data as data_mod
import forecast
import policy
import submission
from schema import ORDER_COL, PRICE_COL, PROB_COL


def locate(dataset: data_mod.Dataset) -> dict[str, object]:
    """Find the history panels the forecasts need, reporting each outcome."""
    found: dict[str, object] = {}
    notes: list[str] = []
    materials = list(dataset.schema.materials)

    def panel(label: str, frame: pd.DataFrame | None, value_hints: tuple[str, ...]):
        if frame is None or frame.empty:
            notes.append(f"{label:<12} FALLBACK  (no table matched)")
            return None
        week_col = data_mod.week_col(frame)
        material_col = forecast.find_col(frame, forecast.MATERIAL_COL_CANDIDATES)
        value_col = forecast.find_col(frame, value_hints)
        if not (week_col and material_col and value_col):
            notes.append(
                f"{label:<12} FALLBACK  (table found, missing "
                f"week/material/value column)"
            )
            return None
        notes.append(f"{label:<12} FOUND     {value_col} by {week_col} x {material_col}")
        return forecast.to_panel(frame, value_col, week_col, material_col)

    found["price"] = panel("price", dataset.prices, ("price", "spot", "cost"))
    found["demand"] = panel("demand", dataset.demand, ("demand", "units", "consumption"))
    found["disruption"] = panel(
        "disruption", dataset.disruptions, ("disrupted", "disruption", "shock", "event")
    )

    country_price = None
    if dataset.prices is not None and not dataset.prices.empty:
        frame = dataset.prices
        week_col = data_mod.week_col(frame)
        material_col = forecast.find_col(frame, forecast.MATERIAL_COL_CANDIDATES)
        country_col = forecast.find_col(frame, forecast.COUNTRY_COL_CANDIDATES)
        value_col = forecast.find_col(frame, ("price", "spot", "cost"))
        if all((week_col, material_col, country_col, value_col)):
            country_price = forecast.to_country_panel(
                frame, value_col, week_col, material_col, country_col
            )
            notes.append(f"{'by-country':<12} FOUND     per-supplier prices")
    if country_price is not None:
        country_price = forecast.carry_forward_country(
            country_price, list(dataset.schema.weeks)
        )
    found["country_price"] = country_price
    found["notes"] = notes
    found["materials"] = materials
    return found


def build_submission(
    dataset: data_mod.Dataset,
    diversification: float,
    cover: float,
    sharpness: float = 3.0,
) -> pd.DataFrame:
    sch = dataset.schema
    horizon = list(sch.weeks)
    located = locate(dataset)
    print("\n".join(located["notes"]))

    materials = located["materials"]
    price_hist = located["price"]
    demand_hist = located["demand"]
    disruption_hist = located["disruption"]

    price = (
        forecast.seasonal_naive(price_hist, horizon)
        if price_hist is not None
        else forecast.flat(1.0, horizon, materials)
    )
    demand_mean = (
        forecast.seasonal_naive(demand_hist, horizon)
        if demand_hist is not None
        else forecast.flat(1.0, horizon, materials)
    )
    demand_sd = (
        forecast.dispersion(demand_hist, horizon)
        if demand_hist is not None
        else forecast.flat(0.0, horizon, materials)
    )
    prob = (
        forecast.base_rate(disruption_hist, horizon)
        if disruption_hist is not None
        else forecast.flat(0.05, horizon, materials)
    )

    costs = config.COSTS
    costs.target_weeks_of_cover = cover
    plan = policy.base_stock_orders(demand_mean, demand_sd, prob, costs)

    shares = policy.risk_tilted_shares(
        schema_countries=list(sch.countries),
        eligibility=dataset.eligibility,
        index=sch.index(),
        country_price=located["country_price"],
        diversification=diversification,
        sharpness=sharpness,
    )

    return submission.build(
        schema=sch,
        prob=prob,
        price=price,
        orders=plan[ORDER_COL],
        shares=shares,
        eligibility=dataset.eligibility,
    )


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--out", type=Path, default=config.OUT_DIR / "submission.csv")
    parser.add_argument(
        "--diversification",
        type=float,
        default=0.5,
        help="0 = cheapest safe supplier, 1 = always split evenly",
    )
    parser.add_argument(
        "--cover", type=float, default=1.0, help="weeks of stock held beyond the lead time"
    )
    parser.add_argument(
        "--sharpness",
        type=float,
        default=3.0,
        help="how hard to favour the cheapest safe supplier before diversifying",
    )
    parser.add_argument("--dry-run", action="store_true", help="validate without writing")
    args = parser.parse_args(argv)

    dataset = data_mod.load()
    print(dataset.summary(), "\n")

    frame = build_submission(
        dataset, args.diversification, args.cover, args.sharpness
    )
    report = submission.validate(frame, dataset.schema, dataset.eligibility)
    print("\n" + str(report))
    if not report.ok:
        return 1

    if args.dry_run:
        print("\ndry run -- nothing written")
        return 0

    path = submission.write(frame, dataset.schema, args.out, dataset.eligibility)
    print(f"\nwrote {path}  ({len(frame)} rows)")
    print(f"service-level target implied by costs: {config.COSTS.critical_ratio:.1%}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile signals.py
"""Estimates fitted from the released weeks, as the EDA established them.

Everything here is a number read off history rather than a modelling choice, so
each one is checkable against `eda/eda_report.md`. The two that matter most:

* `hazard` -- P(disrupted at t+k | clean at t), pooled then shrunk toward
  per-material rates because 31 episodes do not support 10 independent
  far-horizon curves. Firm for the near weeks (P(disrupted next week | clean)
  = 0.0186 on n = 1667) and the reason the baseline's flat unconditional rate
  is wrong at the start of the window.
* `uncond` -- the unconditional per-material rate. Kept because backtesting
  across 123 origins showed the hazard alone is NOT the better predictor: an
  all-clean boundary does not imply a quiet window (in the 24 origins
  resembling week 208 the realised rate averaged 0.314, above the 0.300
  overall). A 50/50 blend beats both ends on mean Brier -- see
  `experiment_blend.py`.
* `country_risk` / `country_cov` -- disruption is a country event. Materials
  sharing a supplier country co-disrupt at r up to 1.000, so the share
  allocation needs the covariance, not just a per-country rate.
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

import config

# A supplier-week counts as hit when capacity is meaningfully below normal or
# the lead time has stretched. Reproduces the material-level `disrupted` label
# at 98.6% agreement, which is what justifies using it as a supplier-level one.
CAP_HIT = 0.9

# Shrinkage weight on the pooled hazard curve. 1.0 = pooled only, 0.0 =
# per-material only. 0.5 keeps the material ordering (steel >> copper) without
# trusting a single material's far-horizon estimate.
HAZARD_POOL_W = 0.5

# Blend of recent and full-history drift for the price random walk.
DRIFT_RECENT_W = 0.5
DRIFT_RECENT_WEEKS = 52


@dataclass
class Fitted:
    materials: list[str]
    countries: list[str]
    last_week: int
    horizon: list[int]
    hazard: pd.DataFrame          # week x material, P(disrupted | clean at boundary)
    uncond: pd.Series             # material -> unconditional P(disrupted)
    clean_at_boundary: pd.Series  # material -> bool
    price_last: pd.Series         # material -> benchmark_price at last_week
    drift: pd.Series              # material -> weekly log drift
    disrupt_premium: float        # price multiple while disrupted
    country_risk: pd.Series       # country -> P(hit)
    country_cov: pd.DataFrame     # country x country covariance of hits
    cap_when_hit: float           # mean capacity ratio on hit weeks
    rel_cost: pd.DataFrame        # material x country, landed cost / row mean
    eligibility: pd.DataFrame     # material x country bool
    lead_weeks: pd.DataFrame      # material x country base lead
    capacity: pd.DataFrame        # material x country normal capacity
    demand: pd.Series             # (week, material) -> units, GIVEN for horizon

    def summary(self) -> str:
        n_clean = int(self.clean_at_boundary.sum())
        return (
            f"fitted on weeks 1-{self.last_week}\n"
            f"  clean at boundary:   {n_clean}/{len(self.materials)} materials\n"
            f"  hazard w{self.horizon[0]}:        {self.hazard.iloc[0].mean():.4f} (mean across materials)\n"
            f"  hazard w{self.horizon[-1]}:        {self.hazard.iloc[-1].mean():.4f}\n"
            f"  disrupt premium:     x{self.disrupt_premium:.4f} on price\n"
            f"  capacity when hit:   {self.cap_when_hit:.4f} of normal\n"
            f"  country risk:        "
            + ", ".join(f"{c}={self.country_risk[c]:.3f}" for c in self.countries)
        )


def _hazard(flags: np.ndarray, max_k: int) -> np.ndarray:
    """P(disrupted at t+k | clean at t) for k = 1..max_k, over one or more series."""
    out = np.zeros(max_k)
    for k in range(1, max_k + 1):
        num = den = 0
        for s in flags:
            n = len(s)
            if n <= k:
                continue
            clean = s[: n - k] == 0
            den += int(clean.sum())
            num += int(s[k:][clean].sum())
        out[k - 1] = num / den if den else np.nan
    return out


def fit(data_dir: Path | None = None, last_week: int | None = None,
        horizon: list[int] | None = None) -> Fitted:
    """Fit every estimate on weeks <= last_week.

    `last_week` and `horizon` are explicit so the same fit powers a backtest:
    pass last_week=182 and horizon=183..208 to score against known outcomes.
    """
    dd = data_dir or config.data_dir()
    L = lambda n: pd.read_csv(dd / f"{n}.csv")
    sup, out, pri, sh, dem = L("suppliers"), L("outcomes"), L("prices"), L("supply_history"), L("demand")

    last_week = last_week or int(out.week.max())
    out = out[out.week <= last_week]
    pri = pri[pri.week <= last_week]
    sh = sh[sh.week <= last_week]
    horizon = horizon or list(range(last_week + 1, last_week + 27))

    materials = sorted(sup.material.unique())
    countries = sorted(sup.country.unique())

    # --- disruption hazard -------------------------------------------------
    piv = out.pivot(index="week", columns="material", values="disrupted").sort_index()
    max_k = max(horizon) - last_week
    pooled = _hazard(np.array([piv[m].to_numpy() for m in materials]), max_k)
    haz = {}
    for m in materials:
        own = _hazard(np.array([piv[m].to_numpy()]), max_k)
        own = np.where(np.isnan(own), pooled, own)
        blended = HAZARD_POOL_W * pooled + (1 - HAZARD_POOL_W) * own
        # A material clean at the boundary follows the hazard; one still
        # disrupted is overwhelmingly likely to stay so (P = 0.923 per week).
        if int(piv[m].loc[last_week]) == 1:
            persist = 0.9231 ** np.arange(1, max_k + 1)
            blended = persist + (1 - persist) * blended
        haz[m] = np.maximum.accumulate(blended)  # hazard is cumulative: never dips
    hazard = pd.DataFrame({m: haz[m][[k - last_week - 1 for k in horizon]] for m in materials},
                          index=horizon)
    clean = pd.Series({m: int(piv[m].loc[last_week]) == 0 for m in materials})
    uncond = piv[materials].mean()

    # --- price -------------------------------------------------------------
    bp = out.pivot(index="week", columns="material", values="benchmark_price").sort_index()
    lr = np.log(bp).diff().dropna()
    drift = DRIFT_RECENT_W * lr.tail(DRIFT_RECENT_WEEKS).mean() + (1 - DRIFT_RECENT_W) * lr.mean()
    o = out.copy()
    o["rel"] = o.benchmark_price / o.groupby("material").benchmark_price.transform("mean")
    g = o.groupby("disrupted").rel.mean()
    premium = float(g.get(1, 1.0) / g.get(0, 1.0)) if len(g) > 1 else 1.0

    # --- country risk and covariance ---------------------------------------
    j = sh.merge(sup[["material", "country", "base_lead_weeks", "normal_capacity_units"]],
                 on=["material", "country"])
    j["cap_ratio"] = j.capacity_units / j.normal_capacity_units
    j["hit"] = (j.cap_ratio < CAP_HIT) | (j.lead_time_weeks > j.base_lead_weeks)
    cw = j.groupby(["week", "country"]).hit.max().unstack().reindex(columns=countries).fillna(False).astype(float)
    country_risk = cw.mean()
    country_cov = cw.cov()
    cap_when_hit = float(j.loc[j.hit, "cap_ratio"].mean())

    # --- static tables ------------------------------------------------------
    elig = sup.pivot_table(index="material", columns="country", values="base_lead_weeks",
                           aggfunc="size").reindex(index=materials, columns=countries).fillna(0) > 0
    lead = sup.pivot_table(index="material", columns="country", values="base_lead_weeks",
                           aggfunc="first").reindex(index=materials, columns=countries)
    cap = sup.pivot_table(index="material", columns="country", values="normal_capacity_units",
                          aggfunc="first").reindex(index=materials, columns=countries).fillna(0.0)
    recent = sh[sh.week > last_week - 13]
    lc = recent.pivot_table(index="material", columns="country", values="landed_cost",
                            aggfunc="mean").reindex(index=materials, columns=countries)
    rel_cost = lc.div(lc.mean(axis=1), axis=0)

    d = dem.set_index(["week", "material"]).demand_units.astype(float)

    return Fitted(materials=materials, countries=countries, last_week=last_week, horizon=horizon,
                  hazard=hazard, uncond=uncond, clean_at_boundary=clean,
                  price_last=bp.loc[last_week],
                  drift=drift, disrupt_premium=premium, country_risk=country_risk,
                  country_cov=country_cov, cap_when_hit=cap_when_hit, rel_cost=rel_cost,
                  eligibility=elig, lead_weeks=lead, capacity=cap, demand=d)


if __name__ == "__main__":
    f = fit()
    print(f.summary())
    print("\nhazard curve (first 5 and last 3 weeks):")
    print(pd.concat([f.hazard.head(5), f.hazard.tail(3)]).to_string(float_format=lambda v: f"{v:.4f}"))


In [ ]:
%%writefile strategy.py
"""Turn the fitted signals into the four submission columns.

Each column follows directly from an EDA finding:

* disruption_prob -- a 50/50 blend of the conditional hazard and the
  unconditional rate. The hazard alone is right about the near weeks (being
  clean at the boundary genuinely caps next-week risk at ~2%) but it assumes
  the calm persists, and across 123 backtested origins that assumption failed
  often enough that pure hazard scored WORSE than the flat unconditional rate
  the baseline uses. The blend minimises mean Brier (0.2130 vs 0.2623 pure
  hazard, 0.2482 pure unconditional).
* price_forecast -- the last observed benchmark price, held flat. Log-returns
  have ~zero autocorrelation, so a random walk's optimal forecast is its last
  value; backtesting confirmed that adding drift makes MAPE worse at every
  origin tried, and the disruption uplift makes it worse on average too.
* share_* -- minimum-variance allocation over the *country* disruption
  covariance with a cost tilt. Materials sharing a country co-disrupt at r up
  to 1.000, so spreading across correlated suppliers is not diversification.
* order_units -- base-stock against demand, which is GIVEN for weeks 209-234,
  inflated for the fraction of a delivery expected not to arrive. A disrupted
  supplier still ships ~0.37 of normal capacity, so the inflation is partial.

Order size and shares are solved TOGETHER, because they are not independent:
a supplier only ships up to its own weekly capacity, so a share of 0.8 on a
supplier holding 1.35 weeks of cover silently truncates an order of 2.5 weeks.
Backtesting the two separately delivered only 0.61-0.73 of what was ordered.
"""

from __future__ import annotations

from itertools import product

import numpy as np
import pandas as pd

from signals import Fitted

# Smallest share any eligible supplier gets, so no row is ever single-sourced.
# A single-sourced row is one shock away from a 5x stockout charge, and the
# baseline submission is single-sourced on all 260 rows.
MIN_SHARE = 0.10
SHARE_GRID = 0.05

# Weight on risk relative to cost in the share objective. Cost is expressed as
# a multiple of the material's mean landed cost, and risk as the standard
# deviation of hit exposure, so the two are on comparable scales.
RISK_AVERSION = 2.0

# Weeks of demand held beyond the lead time. 2.0 chosen by sweeping the
# backtest; the objective is flat within ~2% across the whole grid, so these
# are sensible defaults rather than a tuned edge.
COVER_WEEKS = 2.0
# Extra safety stock as a multiple of the expected disruption shortfall over
# the lead time. 0.0 because COVER_WEEKS already supplies the slack and the
# sweep showed the extra buffer only added holding and purchase cost.
SAFETY = 0.0


# Weight on the conditional hazard against the unconditional rate. 0.5 was
# chosen by sweeping 123 origins (experiment_blend.py); the objective is flat
# between 0.4 and 0.6, so this is not a knife-edge fit.
HAZARD_BLEND = 0.5


def disruption_prob(f: Fitted, blend: float = HAZARD_BLEND) -> pd.Series:
    """Hazard curve blended with the unconditional rate.

    `blend` = 1.0 is pure hazard (right if the window stays calm), 0.0 is the
    flat unconditional rate the baseline submits. The data supports neither
    extreme; see the module docstring.
    """
    haz = f.hazard.stack()
    haz.index.names = ["week", "material"]
    haz = haz.sort_index()
    base = pd.Series([float(f.uncond[m]) for _, m in haz.index], index=haz.index)
    return (blend * haz + (1.0 - blend) * base).clip(0.0, 1.0)


# Weight on the fitted drift and the disruption price premium. Both measured
# worse than zero in backtests, so both default off; kept as levers because a
# later origin may say otherwise.
DRIFT_W = 0.0
UPLIFT_W = 0.0


def price_forecast(f: Fitted, prob: pd.Series, drift_w: float = DRIFT_W,
                   uplift_w: float = UPLIFT_W) -> pd.Series:
    """Last observed benchmark price, held flat by default.

    Log-return autocorrelation is near zero, so the last value is the optimal
    random-walk forecast. `drift_w` and `uplift_w` scale the fitted drift and
    the observed disruption premium; both measured worse than 0 in backtests.
    """
    rows = {}
    uplift = (f.disrupt_premium - 1.0) * uplift_w
    for m in f.materials:
        p0, mu = float(f.price_last[m]), float(f.drift[m]) * drift_w
        for k, week in enumerate(f.horizon, start=1):
            base = p0 * np.exp(mu * k)
            rows[(week, m)] = base * (1.0 + uplift * float(prob[(week, m)]))
    s = pd.Series(rows, dtype=float)
    s.index = pd.MultiIndex.from_tuples(s.index, names=["week", "material"])
    return s.sort_index()


def _simplex(n: int, step: float, floor: float):
    """Enumerate weight vectors on the simplex, each component >= floor."""
    ticks = int(round(1.0 / step))
    lo = int(round(floor / step))
    for combo in product(range(lo, ticks + 1), repeat=n - 1):
        rest = ticks - sum(combo)
        if rest >= lo:
            yield np.array([*combo, rest], dtype=float) * step


def preference(f: Fitted, risk_aversion: float = RISK_AVERSION) -> pd.DataFrame:
    """Target weights per material: minimum-variance with a cost tilt.

    Objective: w.cost + risk_aversion * sqrt(w' Sigma w) over the simplex, every
    eligible supplier at or above MIN_SHARE. Sigma is the covariance of
    country-level disruption, which is what makes this more than an even split:
    KES and SUN share the MERROW lane and co-fail, while AUR has never been hit.

    These are the weights the allocator aims at; capacity may force it off them.
    """
    chosen = {}
    for m in f.materials:
        live = [c for c in f.countries if bool(f.eligibility.loc[m, c])]
        cost = f.rel_cost.loc[m, live].fillna(1.0).to_numpy(float)
        cov = f.country_cov.loc[live, live].to_numpy(float)
        best, best_obj = None, np.inf
        for w in _simplex(len(live), SHARE_GRID, MIN_SHARE):
            obj = float(w @ cost) + risk_aversion * float(np.sqrt(max(w @ cov @ w, 0.0)))
            if obj < best_obj:
                best, best_obj = w, obj
        chosen[m] = dict(zip(live, best))

    frame = pd.DataFrame(0.0, index=f.materials, columns=f.countries)
    for m, w in chosen.items():
        for c, v in w.items():
            frame.loc[m, c] = v
    return frame.div(frame.sum(axis=1), axis=0)


def _water_fill(target: np.ndarray, cap: np.ndarray, qty: float) -> np.ndarray:
    """Allocate `qty` across suppliers, aiming at `target` shares, respecting `cap`.

    Any amount a capped supplier cannot take is redistributed over the
    suppliers that still have slack, in proportion to their target weights.
    This is what stops a preferred-but-small supplier from throttling the
    whole order.
    """
    alloc = np.minimum(target * qty, cap)
    for _ in range(len(target)):
        deficit = qty - alloc.sum()
        if deficit <= 1e-9:
            break
        slack = cap - alloc
        open_ = slack > 1e-9
        if not open_.any():
            break
        weights = target * open_
        weights = weights / weights.sum() if weights.sum() > 0 else open_ / open_.sum()
        alloc = np.minimum(alloc + deficit * weights, cap)
    return alloc


def order_and_shares(f: Fitted, prob: pd.Series, pref: pd.DataFrame,
                     cover: float = COVER_WEEKS, safety: float = SAFETY):
    """Solve order size and sourcing split together, per week and material.

    Returns (orders, shares) where shares is indexed by (week, material), since
    the feasible split depends on how much is being ordered that week.
    """
    orders, share_rows = {}, {}
    short_fall = 1.0 - f.cap_when_hit

    for m in f.materials:
        live = [c for c in f.countries if pref.loc[m, c] > 0]
        target = pref.loc[m, live].to_numpy(float)
        cap = f.capacity.loc[m, live].to_numpy(float)
        leads = f.lead_weeks.loc[m, live].to_numpy(float)
        total_cap = float(cap.sum())
        plan_lead = int(np.ceil(float(np.dot(target, leads))))

        inv = 0.0
        pipeline: dict[int, float] = {}
        for week in f.horizon:
            inv += pipeline.pop(week, 0.0)
            inv = max(0.0, inv - float(f.demand.get((week, m), 0.0)))

            span = [week + i for i in range(1, plan_lead + int(np.ceil(cover)) + 1)]
            need = sum(float(f.demand.get((v, m), 0.0)) for v in span)
            p = float(prob.get((week, m), 0.0))
            per_week = float(f.demand.get((week, m), 0.0))
            target_stock = need + safety * p * short_fall * per_week * plan_lead

            in_transit = sum(pipeline.values())
            gap = max(0.0, target_stock - inv - in_transit)
            arrive_frac = max(0.25, 1.0 - p * short_fall)
            qty = min(gap / arrive_frac, total_cap)

            if qty <= 0:
                # Nothing to order: fall back to the target split so the row
                # still carries a valid, diversified allocation.
                alloc = target * 1.0
            else:
                alloc = _water_fill(target, cap, qty)
            tot = alloc.sum()
            w = alloc / tot if tot > 0 else target

            orders[(week, m)] = qty
            share_rows[(week, m)] = dict(zip(live, w))
            arrive = week + int(np.ceil(float(np.dot(w, leads))))
            pipeline[arrive] = pipeline.get(arrive, 0.0) + qty * arrive_frac

    s_orders = pd.Series(orders, dtype=float)
    s_orders.index = pd.MultiIndex.from_tuples(s_orders.index, names=["week", "material"])

    shares = pd.DataFrame(0.0, index=s_orders.index, columns=f.countries)
    for key, row in share_rows.items():
        for c, v in row.items():
            shares.loc[key, c] = v
    return s_orders.sort_index(), shares.sort_index()


def build(f: Fitted, risk_aversion: float = RISK_AVERSION, cover: float = COVER_WEEKS,
          safety: float = SAFETY, drift_w: float = DRIFT_W, uplift_w: float = UPLIFT_W,
          blend: float = HAZARD_BLEND):
    """All four submission columns. Shares are indexed by (week, material)."""
    prob = disruption_prob(f, blend)
    price = price_forecast(f, prob, drift_w, uplift_w)
    pref = preference(f, risk_aversion)
    orders, shares = order_and_shares(f, prob, pref, cover, safety)
    return prob, price, orders, shares


In [ ]:
%%writefile backtest.py
"""Score a strategy on released weeks, against what actually happened.

Five submissions a day is not a tuning budget, so tuning happens here. Fit on
weeks <= origin, build a submission for the next 26, and play it against the
real supply_history and outcomes. Several origins give several independent
estimates of private-leaderboard behaviour instead of one leaderboard probe.

The rollout uses the actual mechanics the EDA established, not assumptions:
each supplier delivers min(ordered_from_it, its capacity that week), arriving
after that supplier's actual lead time, charged at its actual landed cost.

    python backtest.py                      # three origins, strategy vs baseline
    python backtest.py --origins 182 --grid # sweep the levers at one origin
"""

from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd

import config
import signals
import strategy

STOCKOUT_MULT = 5.0       # stated on the cheat sheet
HOLDING_RATE = 0.01       # CONFIRM: fraction of unit price per week held


def load(dd: Path) -> dict:
    L = lambda n: pd.read_csv(dd / f"{n}.csv")
    sup, out, sh, dem = L("suppliers"), L("outcomes"), L("supply_history"), L("demand")
    sh = sh.merge(sup[["material", "country", "normal_capacity_units"]], on=["material", "country"])
    return {
        "suppliers": sup,
        "outcomes": out.set_index(["week", "material"]),
        "supply": sh.set_index(["week", "material", "country"]),
        "demand": dem.set_index(["week", "material"]).demand_units.astype(float),
        "materials": sorted(sup.material.unique()),
        "countries": sorted(sup.country.unique()),
    }


def play(D: dict, horizon: list[int], orders: pd.Series, shares: pd.DataFrame) -> dict:
    """Roll the orders forward against actual capacity, lead times and costs."""
    purchase = holding = stockout_cost = 0.0
    short_units = demand_units = 0.0
    stockout_weeks = 0
    supply, out, dem = D["supply"], D["outcomes"], D["demand"]

    for m in D["materials"]:
        inv = 0.0
        pipeline: dict[int, float] = {}
        for week in horizon:
            inv += pipeline.pop(week, 0.0)
            want = float(dem.get((week, m), 0.0))
            used = min(inv, want)
            inv -= used
            short = want - used
            demand_units += want
            short_units += short
            if short > 1e-9:
                stockout_weeks += 1

            price = float(out.loc[(week, m), "benchmark_price"])
            stockout_cost += short * price * STOCKOUT_MULT
            holding += inv * price * HOLDING_RATE

            qty = float(orders.get((week, m), 0.0))
            row = shares.loc[(week, m)] if (week, m) in shares.index else shares.loc[m]
            for c in D["countries"]:
                w = float(row.get(c, 0.0))
                if w <= 0 or qty <= 0:
                    continue
                key = (week, m, c)
                if key not in supply.index:
                    continue
                srow = supply.loc[key]
                delivered = min(qty * w, float(srow.capacity_units))
                purchase += delivered * float(srow.landed_cost)
                arrive = week + int(srow.lead_time_weeks)
                pipeline[arrive] = pipeline.get(arrive, 0.0) + delivered

    hhi = float((shares ** 2).sum(axis=1).mean())
    return {
        "total_cost": purchase + holding + stockout_cost,
        "purchase": purchase, "holding": holding, "stockout": stockout_cost,
        "service_level": 1.0 - short_units / demand_units if demand_units else 1.0,
        "stockout_weeks": stockout_weeks,
        "mean_hhi": hhi,
        "suppliers_used": float((shares > 0.01).sum(axis=1).mean()),
    }


def baseline(D: dict, f: signals.Fitted) -> tuple[pd.Series, pd.Series, pd.Series, pd.DataFrame]:
    """The naive submission: unconditional rate, last price, order = demand, single source.

    This is what sample_submission.csv does -- its lithium disruption_prob of
    0.2067 is exactly lithium's 208-week mean, and every row is single-sourced.
    """
    out = D["outcomes"].reset_index()
    tr = out[out.week <= f.last_week]
    rate = tr.groupby("material").disrupted.mean()
    idx = pd.MultiIndex.from_product([f.horizon, f.materials], names=["week", "material"])
    prob = pd.Series([rate[m] for _, m in idx], index=idx)
    price = pd.Series([float(f.price_last[m]) for _, m in idx], index=idx)
    orders = pd.Series([float(f.demand.get((w, m), 0.0)) for w, m in idx], index=idx)

    shares = pd.DataFrame(0.0, index=idx, columns=f.countries)
    for m in f.materials:
        live = [c for c in f.countries if bool(f.eligibility.loc[m, c])]
        cheapest = f.rel_cost.loc[m, live].astype(float).idxmin()
        shares.loc[(slice(None), m), cheapest] = 1.0
    return prob, price, orders, shares


def prediction_scores(D: dict, f: signals.Fitted, prob: pd.Series, price: pd.Series) -> dict:
    out = D["outcomes"]
    idx = [(w, m) for w in f.horizon for m in f.materials if (w, m) in out.index]
    actual_d = np.array([float(out.loc[k, "disrupted"]) for k in idx])
    actual_p = np.array([float(out.loc[k, "benchmark_price"]) for k in idx])
    pp = np.array([float(prob.get(k, 0.0)) for k in idx])
    fp = np.array([float(price.get(k, 0.0)) for k in idx])
    return {
        "brier": float(np.mean((pp - actual_d) ** 2)),
        "price_mape": float(np.mean(np.abs((fp - actual_p) / actual_p))),
        "price_rmse": float(np.sqrt(np.mean((fp - actual_p) ** 2))),
        "realised_disruption_rate": float(actual_d.mean()),
    }


def run_origin(D: dict, dd: Path, origin: int, risk: float, cover: float, safety: float) -> pd.DataFrame:
    horizon = list(range(origin + 1, origin + 27))
    f = signals.fit(dd, last_week=origin, horizon=horizon)
    rows = []
    for label, (prob, price, orders, shares) in {
        "strategy": strategy.build(f, risk, cover, safety),
        "baseline": baseline(D, f),
    }.items():
        r = {"origin": origin, "variant": label}
        r.update(prediction_scores(D, f, prob, price))
        r.update(play(D, horizon, orders, shares))
        r["ordered"] = float(orders.sum())
        rows.append(r)
    return pd.DataFrame(rows)


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--origins", type=int, nargs="*", default=[156, 169, 182])
    ap.add_argument("--risk-aversion", type=float, default=strategy.RISK_AVERSION)
    ap.add_argument("--cover", type=float, default=strategy.COVER_WEEKS)
    ap.add_argument("--safety", type=float, default=strategy.SAFETY)
    ap.add_argument("--grid", action="store_true", help="sweep levers at the given origins")
    args = ap.parse_args(argv)

    dd = config.data_dir()
    D = load(dd)

    if args.grid:
        rows = []
        for origin in args.origins:
            horizon = list(range(origin + 1, origin + 27))
            f = signals.fit(dd, last_week=origin, horizon=horizon)
            for risk in (0.5, 1.0, 2.0, 4.0):
                for cover in (0.0, 1.0, 2.0):
                    for safety in (0.0, 1.0, 2.0):
                        prob, price, orders, shares = strategy.build(f, risk, cover, safety)
                        r = {"origin": origin, "risk": risk, "cover": cover, "safety": safety}
                        r.update(play(D, horizon, orders, shares))
                        rows.append(r)
        g = pd.DataFrame(rows)
        agg = g.groupby(["risk", "cover", "safety"]).agg(
            total_cost=("total_cost", "mean"), stockout=("stockout", "mean"),
            service=("service_level", "mean"), hhi=("mean_hhi", "mean")).sort_values("total_cost")
        print("lever sweep, averaged over origins (lower total_cost is better):")
        print(agg.to_string(float_format=lambda v: f"{v:,.4f}"))
        return 0

    frames = [run_origin(D, dd, o, args.risk_aversion, args.cover, args.safety) for o in args.origins]
    res = pd.concat(frames, ignore_index=True)
    cols = ["origin", "variant", "brier", "price_mape", "total_cost", "stockout",
            "service_level", "stockout_weeks", "mean_hhi", "ordered"]
    print(res[cols].to_string(index=False, float_format=lambda v: f"{v:,.4f}"))

    print("\n" + "=" * 70)
    print("strategy vs baseline, averaged over origins")
    print("=" * 70)
    piv = res.groupby("variant")[["brier", "price_mape", "total_cost", "stockout",
                                  "service_level", "mean_hhi"]].mean()
    print(piv.to_string(float_format=lambda v: f"{v:,.4f}"))
    s, b = piv.loc["strategy"], piv.loc["baseline"]
    print(f"\nBrier       {b.brier:.4f} -> {s.brier:.4f}   ({(1 - s.brier/b.brier):+.1%})")
    print(f"price MAPE  {b.price_mape:.4f} -> {s.price_mape:.4f}   ({(1 - s.price_mape/b.price_mape):+.1%})")
    print(f"total cost  {b.total_cost:,.0f} -> {s.total_cost:,.0f}   ({(1 - s.total_cost/b.total_cost):+.1%})")
    print(f"service     {b.service_level:.4f} -> {s.service_level:.4f}")
    print(f"mean HHI    {b.mean_hhi:.4f} -> {s.mean_hhi:.4f}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## Load the data


In [ ]:
!ls -R /kaggle/input 2>/dev/null | head -40 || echo "NOTHING ATTACHED"

In [ ]:
import os; os.environ["DSDG_DATA_DIR"] = "/kaggle/input/competitions/dsdg-hackathon-2026-supply-chain-disruptor"

In [ ]:
import pandas as pd, numpy as np
import sys, os
from pathlib import Path

# %%writefile wrote the modules into the notebook's working directory.
sys.path.insert(0, os.getcwd())

import config, data as data_mod, forecast, policy, simulator, submission, run
import signals, strategy, backtest

dataset = data_mod.load()
print(dataset.summary())
print()
print(f"output dir:            {config.OUT_DIR}")
print(f"implied service level: {config.COSTS.critical_ratio:.2%}")
unconfirmed = config.COSTS.audit()
if unconfirmed:
    print(f"UNCONFIRMED constants: {', '.join(unconfirmed)}")
    print("  -> read these off the competition description before trusting a local score")


## Build the submission


In [ ]:
# Levers, all chosen by backtest (backtest.py), not by guess. The sweep is
# flat within ~2% across the whole grid, so these are defaults rather than an
# edge -- re-sweep once phase 2 lands and weeks 209-221 have known outcomes.
RISK_AVERSION = 2.0   # weight on disruption variance vs landed cost in the share split
COVER = 2.0           # weeks of demand held beyond the lead time
SAFETY = 0.0          # extra buffer on top of COVER; the sweep preferred none
BLEND = 0.5           # 1.0 = pure conditional hazard, 0.0 = flat unconditional rate

f = signals.fit(dataset.dir)
print(f.summary())

prob, price, orders, shares = strategy.build(
    f, risk_aversion=RISK_AVERSION, cover=COVER, safety=SAFETY, blend=BLEND)

idx = dataset.schema.index()
frame = submission.build(
    schema=dataset.schema, prob=prob, price=price, orders=orders,
    shares=shares.reindex(index=idx, columns=list(dataset.schema.countries)),
    eligibility=dataset.eligibility)

report = submission.validate(frame, dataset.schema, dataset.eligibility)
print()
print(report)
print()
print(f"mean HHI {float((shares ** 2).sum(axis=1).mean()):.4f}  (baseline is 1.000)")
print(f"disruption_prob w{f.horizon[0]} {prob.groupby('week').mean().iloc[0]:.4f}"
      f" -> w{f.horizon[-1]} {prob.groupby('week').mean().iloc[-1]:.4f}")


In [ ]:
# submission.write re-validates and refuses to write an invalid file, so a
# green cell here means the CSV is accepted.
path = submission.write(
    frame, dataset.schema, config.OUT_DIR / "submission.csv", dataset.eligibility
)
print(f"wrote {path}  ({len(frame)} rows)")
frame.head()


## Validating a policy — the part that actually matters

Five submissions a day is 15 probes total against the component deciding 56%
of placement, so the tuning happens here instead.

After Saturday noon the data covers weeks 1–221, which includes the whole
public leaderboard window (209–221). Those weeks go **in-sample** — scoring
them perfectly teaches nothing about the private weeks 222–234. What they are
good for is validation with known outcomes: build a submission for 209–221,
play it through `simulator.rollout`, and sweep the three levers against a real
cost ledger.

Two consequences worth repeating: tune locally rather than on the leaderboard,
and **select the two final submissions by hand** — Kaggle's default picks best
public score, which after Saturday noon selects for whatever overfit the
in-sample window hardest.


In [ ]:
# backtest.py already does this against the real supply history: it fits on
# weeks <= origin and plays the resulting submission through the actual
# capacity, lead times and landed costs of the next 26 weeks.
#
# Phase 1 origins (fit <= origin, score origin+1 .. origin+26):
D = backtest.load(dataset.dir)
res = pd.concat([backtest.run_origin(D, dataset.dir, o, RISK_AVERSION, COVER, SAFETY)
                 for o in (156, 169, 182)], ignore_index=True)
print(res[["origin","variant","brier","price_mape","total_cost","stockout",
           "service_level","mean_hhi"]].to_string(index=False))

# Once phase 2 lands, add 195 and 208 as origins -- origin 195 scores weeks
# 196-221, which is the public window with known outcomes, and is the closest
# proxy for the private weeks you will get.


## Before submitting

- [ ] every `FOUND`/`FALLBACK` line above reads FOUND — a FALLBACK means that
      table was not located and the orders are placeholders
- [ ] the unconfirmed cost constants have been checked against the competition
      description (office hours Sat 10–5 is the place to ask)
- [ ] `submission valid` printed, with no eligibility warning
- [ ] levers tuned against weeks 209–221, not against the public leaderboard
- [ ] two final submissions selected **by hand** before Sunday noon
- [ ] video uploaded — 20% of placement, same deadline
